In [1]:
from pathlib import Path
import pandas as pd

file_path = str(Path.cwd()/"resourse/faq_data.csv")

def ingest_faq():

    data = pd.read_csv(file_path)
    return data.head()


In [2]:
print(ingest_faq())

                                       question  \
0    What is the return policy of the products?   
1  Do I get discount with the HDFC credit card?   
2                     How can I track my order?   
3            What payment methods are accepted?   
4    How long does it take to process a refund?   

                                              answer  
0  You can return products within 30 days of deli...  
1  Yes, HDFC credit card users get a 10% discount...  
2  You can track your order using the tracking li...  
3  We accept credit/debit cards, net banking, UPI...  
4  Refunds are processed within 5-7 business days...  


In [3]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_groq import ChatGroq
import chromadb
from dotenv import load_dotenv
load_dotenv(".env.txt")

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
collection_name = "faq"
ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8713.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def initialize_components(path):
    global collection
    
    if collection_name not in [c.name for c in chroma_client.list_collections()]:
        
        collection = chroma_client.get_or_create_collection(
            name=collection_name,
            embedding_function=ef
        )

        data = pd.read_csv(path)

        docs = data['question'].to_list()
        metadata = [{"answer": ans} for ans in data['answer'].to_list()]
        ids = [f"id{i}" for i in range(len(docs))]

        collection.add(
            documents=docs,
            metadatas=metadata,
            ids=ids
        )

        return "Collection successfully added to ChromaDB."
    
    else:
        # LOAD existing collection
        collection = chroma_client.get_collection(
            name=collection_name,
            embedding_function=ef
        )
        return f"Collection {collection_name} loaded."
    
call = initialize_components(file_path)


In [6]:
llm = ChatGroq(model="openai/gpt-oss-120b",
                temperature=0.3, 
                max_tokens=150)

In [14]:
def get_relevent_query(query):

    collection = chroma_client.get_collection(name=collection_name)

    result = collection.query(query_texts=[query],
                              n_results=2)
    return result
query = "what is your returning policy"

get_relevent_query(query)

{'ids': [['id0', 'id4']],
 'embeddings': None,
 'documents': [['What is the return policy of the products?',
   'How long does it take to process a refund?']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'answer': 'You can return products within 30 days of delivery.'},
   {'answer': 'Refunds are processed within 5-7 business days after the return is received.'}]],
 'distances': [[0.20653611421585083, 0.6786532402038574]]}

In [16]:
def get_embaddings(query):

    context = " ".join(r["answer"] for r in query["metadatas"][0])

    return context

context = get_embaddings(get_relevent_query(query))

context

'You can return products within 30 days of delivery. Refunds are processed within 5-7 business days after the return is received.'

In [9]:
import re

def get_final_result(query, context):

    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0.3, 
        max_tokens=150
    )
    
    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

If the answer is not in the context, say "I don't know contact our team for further details".
"""
    
    response = llm.invoke(prompt)

    return re.sub(r'\s+', ' ', response.content)

In [10]:
path = initialize_components(file_path)
query = "what if i found defective piece"
result = get_relevent_query(query) # just for fetching the data 
context = get_embaddings(result)

get_final_result(query, context)

'If you discover a defective piece, contact our support team within 48 hours to arrange a replacement or a refund.'